# 🛫 MÓDULO 1: Exploración de Datos
## Predicción de Retrasos Aéreos - American Regional Airways

**Objetivo:** Conocer el dataset y calcular estadísticas descriptivas básicas.

---

## 1.1 Configuración del Ambiente

Este código detecta automáticamente si estás en **Google Colab** o en **Docker**.

In [1]:
import sys

# Detectar ambiente
EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    print("📍 Ejecutando en Google Colab")
    !pip install pyspark -q
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_DATOS = '/content/drive/MyDrive/ML_Vuelos/data/'  # Ajustar según tu Drive
else:
    print("🐳 Ejecutando en Docker")
    RUTA_DATOS = '../data/'

print(f"\n📂 Ruta de datos: {RUTA_DATOS}")

🐳 Ejecutando en Docker

📂 Ruta de datos: ../data/


In [2]:
# Inicializar Spark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Vuelos_Modulo1") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"✅ Spark {spark.version} iniciado")

✅ Spark 3.5.0 iniciado


## 1.2 Cargar el Dataset

In [3]:
# Cargar datos de vuelos
df = spark.read.csv(
    RUTA_DATOS + "flights_2015_full.csv",
    header=True,
    inferSchema=True
)

print(f"✈️ Dataset cargado: {df.count():,} vuelos")
print(f"📊 Columnas: {len(df.columns)}")

✈️ Dataset cargado: 5,332,914 vuelos
📊 Columnas: 8


In [4]:
# Ver estructura
df.printSchema()

root
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEP_HOUR: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- DEP_DEL15: integer (nullable = true)



In [5]:
# Primeras filas
df.show(5)

+-----+-----------+-------+------+----+--------+--------+---------+
|MONTH|DAY_OF_WEEK|AIRLINE|ORIGIN|DEST|DEP_HOUR|DISTANCE|DEP_DEL15|
+-----+-----------+-------+------+----+--------+--------+---------+
|    1|          4|     AS|   ANC| SEA|       0|    1448|        0|
|    1|          4|     AA|   LAX| PBI|       0|    2330|        0|
|    1|          4|     US|   SFO| CLT|       0|    2296|        0|
|    1|          4|     AA|   LAX| MIA|       0|    2342|        0|
|    1|          4|     AS|   SEA| ANC|       0|    1448|        0|
+-----+-----------+-------+------+----+--------+--------+---------+
only showing top 5 rows



## 1.3 Estadísticas Descriptivas

In [8]:
# Estadísticas numéricas
df.describe().show()

+-------+------------------+------------------+-------+-------+-------+-----------------+-----------------+-------------------+
|summary|             MONTH|       DAY_OF_WEEK|AIRLINE| ORIGIN|   DEST|         DEP_HOUR|         DISTANCE|          DEP_DEL15|
+-------+------------------+------------------+-------+-------+-------+-----------------+-----------------+-------------------+
|  count|           5332914|           5332914|5332914|5332914|5332914|          5332914|          5332914|            5332914|
|   mean| 6.207209979384629| 3.919178895440654|   NULL|   NULL|   NULL|13.03415468541214|822.8955314111572| 0.1869917647275017|
| stddev|3.3838067324500085|1.9936350491708443|   NULL|   NULL|   NULL|4.830332212803467|607.7991894508397|0.38990495400634334|
|    min|                 1|                 1|     AA|    ABE|    ABE|                0|               21|                  0|
|    max|                12|                 7|     WN|    YUM|    YUM|               23|             49

In [9]:
# Calcular tasa de retrasos
from pyspark.sql.functions import col, sum as spark_sum, count, round as spark_round

total_vuelos = df.count()
vuelos_retrasados = df.filter(col("DEP_DEL15") == 1).count()
tasa_retrasos = (vuelos_retrasados / total_vuelos) * 100

print("="*50)
print("📊 RESUMEN DEL DATASET")
print("="*50)
print(f"Total de vuelos:      {total_vuelos:>12,}")
print(f"Vuelos retrasados:    {vuelos_retrasados:>12,}")
print(f"Vuelos puntuales:     {total_vuelos - vuelos_retrasados:>12,}")
print(f"Tasa de retrasos:     {tasa_retrasos:>11.2f}%")
print("="*50)

📊 RESUMEN DEL DATASET
Total de vuelos:         5,332,914
Vuelos retrasados:         997,211
Vuelos puntuales:        4,335,703
Tasa de retrasos:           18.70%


## 1.4 Exploración por Variables

In [10]:
# Retrasos por MES
print("📅 RETRASOS POR MES")
df.groupBy("MONTH") \
    .agg(
        count("*").alias("total"),
        spark_sum("DEP_DEL15").alias("retrasados"),
        spark_round(spark_sum("DEP_DEL15") / count("*") * 100, 2).alias("tasa_%")
    ) \
    .orderBy("MONTH") \
    .show(12)

📅 RETRASOS POR MES
+-----+------+----------+------+
|MONTH| total|retrasados|tasa_%|
+-----+------+----------+------+
|    1|469968|     90978| 19.36|
|    2|429191|     91288| 21.27|
|    3|504312|     95585| 18.95|
|    4|485151|     79889| 16.47|
|    5|496993|     89816| 18.07|
|    6|503897|    116858| 23.19|
|    7|520718|    109884|  21.1|
|    8|510536|     96519| 18.91|
|    9|464946|     58488| 12.58|
|   11|467972|     69784| 14.91|
|   12|479230|     98122| 20.47|
+-----+------+----------+------+



In [11]:
# Retrasos por DÍA DE LA SEMANA
print("📆 RETRASOS POR DÍA (1=Lunes, 7=Domingo)")
df.groupBy("DAY_OF_WEEK") \
    .agg(
        count("*").alias("total"),
        spark_sum("DEP_DEL15").alias("retrasados"),
        spark_round(spark_sum("DEP_DEL15") / count("*") * 100, 2).alias("tasa_%")
    ) \
    .orderBy("DAY_OF_WEEK") \
    .show(7)

📆 RETRASOS POR DÍA (1=Lunes, 7=Domingo)
+-----------+------+----------+------+
|DAY_OF_WEEK| total|retrasados|tasa_%|
+-----------+------+----------+------+
|          1|799248|    158486| 19.83|
|          2|780858|    144666| 18.53|
|          3|790991|    144258| 18.24|
|          4|789594|    155958| 19.75|
|          5|779708|    147841| 18.96|
|          6|637814|    105701| 16.57|
|          7|754701|    140301| 18.59|
+-----------+------+----------+------+



In [12]:
# Retrasos por HORA DE SALIDA
print("🕐 RETRASOS POR HORA DE SALIDA")
df.groupBy("DEP_HOUR") \
    .agg(
        count("*").alias("total"),
        spark_sum("DEP_DEL15").alias("retrasados"),
        spark_round(spark_sum("DEP_DEL15") / count("*") * 100, 2).alias("tasa_%")
    ) \
    .orderBy("DEP_HOUR") \
    .show(24)

🕐 RETRASOS POR HORA DE SALIDA
+--------+------+----------+------+
|DEP_HOUR| total|retrasados|tasa_%|
+--------+------+----------+------+
|       0| 13720|      2339| 17.05|
|       1|  4920|       874| 17.76|
|       2|  1343|       230| 17.13|
|       3|   729|       154| 21.12|
|       4|   528|       127| 24.05|
|       5|108653|      6414|   5.9|
|       6|373324|     25009|   6.7|
|       7|359705|     32281|  8.97|
|       8|349326|     38893| 11.13|
|       9|321313|     42714| 13.29|
|      10|339876|     51551| 15.17|
|      11|327369|     54459| 16.64|
|      12|325368|     59017| 18.14|
|      13|333142|     64411| 19.33|
|      14|301763|     65223| 21.61|
|      15|336908|     75599| 22.44|
|      16|306418|     72842| 23.77|
|      17|357373|     89225| 24.97|
|      18|306560|     83310| 27.18|
|      19|303329|     83506| 27.53|
|      20|238717|     67255| 28.17|
|      21|172831|     46260| 26.77|
|      22|109059|     26895| 24.66|
|      23| 40640|      8623| 21.22

In [13]:
# Retrasos por AEROLÍNEA
print("✈️ RETRASOS POR AEROLÍNEA (ordenado por tasa)")
df.groupBy("AIRLINE") \
    .agg(
        count("*").alias("total"),
        spark_sum("DEP_DEL15").alias("retrasados"),
        spark_round(spark_sum("DEP_DEL15") / count("*") * 100, 2).alias("tasa_%")
    ) \
    .orderBy(col("tasa_%").desc()) \
    .show(15)

✈️ RETRASOS POR AEROLÍNEA (ordenado por tasa)
+-------+-------+----------+------+
|AIRLINE|  total|retrasados|tasa_%|
+-------+-------+----------+------+
|     NK| 107171|     29731| 27.74|
|     UA| 469829|    114059| 24.28|
|     F9|  82735|     19670| 23.77|
|     B6| 245135|     53697| 21.91|
|     WN|1157339|    251000| 21.69|
|     MQ| 272650|     53951| 19.79|
|     VX|  56439|     10170| 18.02|
|     AA| 648694|    114293| 17.62|
|     EV| 526249|     91185| 17.33|
|     OO| 539545|     90918| 16.85|
|     US| 198715|     29144| 14.67|
|     DL| 800329|    116765| 14.59|
|     AS| 158054|     17435| 11.03|
|     HA|  70030|      5193|  7.42|
+-------+-------+----------+------+



In [14]:
# Top 10 aeropuertos ORIGEN con más retrasos
print("🛫 TOP 10 AEROPUERTOS ORIGEN CON MÁS RETRASOS")
df.groupBy("ORIGIN") \
    .agg(
        count("*").alias("total"),
        spark_sum("DEP_DEL15").alias("retrasados"),
        spark_round(spark_sum("DEP_DEL15") / count("*") * 100, 2).alias("tasa_%")
    ) \
    .filter(col("total") > 10000) \
    .orderBy(col("tasa_%").desc()) \
    .show(10)

🛫 TOP 10 AEROPUERTOS ORIGEN CON MÁS RETRASOS
+------+------+----------+------+
|ORIGIN| total|retrasados|tasa_%|
+------+------+----------+------+
|   MDW| 80886|     19693| 24.35|
|   BWI| 86079|     20873| 24.25|
|   ORD|285884|     67810| 23.72|
|   DAL| 59699|     14050| 23.53|
|   HOU| 52042|     12183| 23.41|
|   EWR|101772|     23552| 23.14|
|   MIA| 69341|     16022| 23.11|
|   DEN|196055|     44426| 22.66|
|   PBI| 22573|      4967|  22.0|
|   LGA| 99605|     21529| 21.61|
+------+------+----------+------+
only showing top 10 rows



---
## ✅ CHECKPOINT MÓDULO 1

Antes de continuar, verifica que puedes responder:

1. ¿Cuántos vuelos tiene el dataset?
2. ¿Cuál es la tasa general de retrasos?
3. ¿Qué mes tiene más retrasos?
4. ¿Qué aerolínea tiene la peor tasa?

**Siguiente:** → `Modulo2_Preparacion.ipynb`

---
## ✅ Respuestas MÓDULO 1

Antes de continuar, verifica que puedes responder:

1. ¿Cuántos vuelos tiene el dataset?   Rta/tiene 5,332,914 vuelos
2. ¿Cuál es la tasa general de retrasos?   Rta/ 18.70%
3. ¿Qué mes tiene más retrasos?  Rta/ el mes de junio con una tasa del 23.19%
4. ¿Qué aerolínea tiene la peor tasa? Rta/ Aerolinea NK con una tasa del 27.74%

**Siguiente:** → `Modulo2_Preparacion.ipynb`